# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will:
- List all record sets by `@id` and name.
- For each record set, list their fields (columns) and their IDs.

In [ ]:
# List all record sets
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"\nRecordSet: @id={rs['@id']}, name={rs.get('name', '<no name>')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            fname = field.get('name', '<no name>') if isinstance(field, dict) else '<unknown>'
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"    Field/Column: @id={fid}, name={fname}")
    else:
        print("    (No fields listed for this record set)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

You must use the record set and field `@id`s from the overview above when referencing data, as per the Croissant specification.

In [ ]:
# Example: Extract data from all record sets
# (Replace these IDs with those listed above for your analysis)
# Example assumes one main record set containing protein data.

# Collect all record set IDs
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record set IDs:", record_set_ids)
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nExtracting records from: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For further EDA, pick the main record set (likely first one if unsure):
main_record_set_id = record_set_ids[0]
print(f"\nMain record set: {main_record_set_id}")
df = dataframes[main_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we'll select a numeric column by its `@id`, perform filtering and normalization, then group by a categorical field.

In [ ]:
# Example field @ids from schema (replace with those found from Data Overview):
# Let's try to infer numeric columns from df:
print("Available columns:", df.columns.tolist())

# Try to find a numeric field such as 'molecular_weight' or 'coverage_percent'
possible_numeric = [col for col in df.columns if ('weight' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower() or 'peptide_count' in col.lower())]
if not possible_numeric:
    numeric_field = df.select_dtypes('number').columns[0]
else:
    numeric_field = possible_numeric[0]
print(f"Selected numeric field: {numeric_field}")

# Threshold example (customize as appropriate)
threshold = df[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
if threshold is not None:
    filtered_df = df[df[numeric_field] > threshold].copy()
else:
    filtered_df = df.copy()

print(f"Filtered for {numeric_field} > {threshold} ({len(filtered_df)} records):")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized column added:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a category, e.g., 'description' column (or, if alphabetically, first string column longer than 3 unique values not id or acc)
possible_groups = [col for col in df.columns if df[col].dtype == object and len(df[col].unique()) < 50 and not col.lower().endswith('id')]
group_field = possible_groups[0] if possible_groups else df.columns[0]  # fallback
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing mean of {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Counts")
plt.show()

if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset using `mlcroissant`, explored its structure, performed elementary cleaning and normalization steps, and conducted initial visualizations.

**Key findings:**
- The dataset provides detailed protein-level quantification and annotation for human mast cell-derived extracellular vesicles.
- Numeric data fields (such as molecular weight, coverage, etc.) are available for analysis.
- Filtering and normalizing these fields helps prepare the data for downstream exploration, such as identifying outlier proteins or comparing abundances between sample groups.

Further analysis may include: differential expression, advanced clustering, or cross-referencing with UniProt annotations for biological insight.

---
For reproducible workflow, always reference field and record set `@id`s from the Croissant metadata.